## Building the fundamentals loader

Goal: for every entity in the point in time universe (per `universe_spans`), retrieve the raw
financial statement figures needed for the eight fundamentals-based JKP themes plus shares
outstanding (value, profitability, quality, profit growth, investment, accruals, debt
issuance, low leverage, and size, per `notebooks/logs/factor_suite_design.md`), each tagged
with the date the figure became public, not merely the period it describes, so a backtest can
use only what was actually known on a given date.

Start small: pull one well known, consistently tagged company's raw XBRL company facts from
EDGAR, with no interpretation yet, before designing anything around the harder questions: tag
standardization across filers, and whether restated values can be reconstructed point in time
from the filed history rather than only ever showing the current, restated figure.

## Part 1: the vendor's raw shape

Before designing anything, look at what EDGAR's `companyfacts` API actually returns for one
familiar filer, full history, no interpretation. Apple (CIK 320193) is a large, mature filer,
a reasonable first case before checking a smaller or less consistently tagged company.

In [1]:
import requests

SEC_HEADERS = {"User-Agent": "capm-portfolio-research kevin (contact: hongxianl957@gmail.com)"}

AAPL_CIK = 320193
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{AAPL_CIK:010d}.json"
r = requests.get(url, headers=SEC_HEADERS, timeout=15)
r.raise_for_status()
facts = r.json()

print("entityName:", facts["entityName"])
print("top level fact taxonomies:", list(facts["facts"].keys()))
print("us-gaap tag count:", len(facts["facts"]["us-gaap"]))
print("dei tag count:", len(facts["facts"]["dei"]))

entityName: Apple Inc.
top level fact taxonomies: ['dei', 'us-gaap']
us-gaap tag count: 503
dei tag count: 2


In [2]:
# The concepts needed across the eight fundamentals-based JKP themes plus shares
# outstanding, distilled from the data-dependency map in
# notebooks/logs/factor_suite_design.md. Defined once, up front, rather than each
# factor reaching into raw EDGAR tag names itself: the loader's job is "fetch these
# facts", the factor's job is "turn an already-fetched value into a score", and this
# list is the seam between the two.
NEEDED_US_GAAP_TAGS = [
    "Assets",                                       # investment; low leverage; profitability denominator
    "Liabilities",                                   # low leverage
    "StockholdersEquity",                            # value (book/price); profitability (ROE); low leverage
    "NetIncomeLoss",                                 # value (E/P); profitability; quality; profit growth
    "Revenues",                                      # value (S/P); profitability (gross margin)
    "GrossProfit",                                    # profitability
    "NetCashProvidedByUsedInOperatingActivities",     # value (CF/P); accruals
    "LongTermDebtNoncurrent",                         # debt issuance; low leverage
    "LongTermDebtCurrent",                            # debt issuance; low leverage
]
NEEDED_DEI_TAGS = [
    "EntityCommonStockSharesOutstanding",  # size; denominator for every "/price" ratio above
]

# A tag missing here does not prove Apple lacks the concept. It may report the same
# economic fact under a different, non-standardized tag name (e.g. "SalesRevenueNet"
# instead of "Revenues"). This pass only checks the exact names assumed above; a miss
# is a prompt to go looking for a synonym next, not proof of absence.
missing_us_gaap = [tag for tag in NEEDED_US_GAAP_TAGS if tag not in facts["facts"]["us-gaap"]]
missing_dei = [tag for tag in NEEDED_DEI_TAGS if tag not in facts["facts"]["dei"]]

print("us-gaap tags missing from Apple's filings:", missing_us_gaap)
print("dei tags missing from Apple's filings:", missing_dei)


us-gaap tags missing from Apple's filings: []
dei tags missing from Apple's filings: []


In [3]:
# Pulled into a function now that it's called for more than one company: the fetch
# itself (URL shape, header, error handling) is identical every time, only the CIK
# changes. Mirrors why fetch_cik_prices in src/loaders/prices.py exists as a function
# rather than being repeated inline per ticker.
def fetch_company_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    r = requests.get(url, headers=SEC_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

# Costco: a large, older filer, included as a second "clean" data point, to check
# whether Apple's clean result was a fluke of one company or holds more broadly.
# DoorDash: already present in data/raw/prices/ (CIK 1792789), IPO'd December 2020,
# well after the 2018 accounting-standard change (ASC 606) that introduced the newer
# revenue-recognition tag. A plausible case for a filer that never reported under the
# older "Revenues" tag name at all, rather than one that's simply missing data.
companies_to_check = [("Apple", 320193), ("Costco", 909832), ("DoorDash", 1792789)]

for name, cik in companies_to_check:
    facts = fetch_company_facts(cik)
    missing_gaap = [t for t in NEEDED_US_GAAP_TAGS if t not in facts["facts"].get("us-gaap", {})]
    missing_dei = [t for t in NEEDED_DEI_TAGS if t not in facts["facts"].get("dei", {})]
    print(name, "missing us-gaap:", missing_gaap, "| missing dei:", missing_dei)


Apple missing us-gaap: [] | missing dei: []
Costco missing us-gaap: [] | missing dei: []
DoorDash missing us-gaap: ['Revenues', 'GrossProfit', 'LongTermDebtNoncurrent', 'LongTermDebtCurrent'] | missing dei: ['EntityCommonStockSharesOutstanding']


In [4]:
# Revenue's tag name changed across filers around the 2018 ASC 606 accounting
# standard update: older filers report "Revenues", DoorDash (and other filers that
# never used the pre-2018 tag) report only "RevenueFromContractWithCustomerExcludingAssessedTax".
# Rather than hardcoding one tag name per concept, each concept maps to an ordered
# list of tag names to try, first match wins. This is the fundamentals-loader
# equivalent of the ticker-format translation prices.py already does at its own
# vendor boundary (BRK.B vs BRK-B).
TAG_ALIASES = {
    "revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
}

def resolve_tag(facts, taxonomy, concept):
    # Returns the first alias actually present in this company's facts, or None
    # if the company reports the concept under some third name not yet seen.
    available = facts["facts"].get(taxonomy, {})
    for candidate in TAG_ALIASES[concept]:
        if candidate in available:
            return candidate
    return None

for name, cik in companies_to_check:
    facts = fetch_company_facts(cik)
    print(name, "resolves revenue to:", resolve_tag(facts, "us-gaap", "revenue"))


Apple resolves revenue to: Revenues
Costco resolves revenue to: Revenues
DoorDash resolves revenue to: RevenueFromContractWithCustomerExcludingAssessedTax


In [5]:
# So far every check has only been about tag names, never the actual data points
# behind a tag. The other open question from when this notebook started
# (notebooks/logs/factor_suite_design.md) is whether EDGAR's raw facts retain
# restated values as separate historical entries, each with their own "filed" date,
# rather than silently overwriting the original number the way a data vendor that
# only shows "the current figure" would. That distinction is what the README's
# existing point-in-time-versus-restated limitation is really about, and it has not
# yet been checked directly against the primary source.
apple_facts = fetch_company_facts(320193)
net_income_points = apple_facts["facts"]["us-gaap"]["NetIncomeLoss"]["units"]["USD"]

print("total NetIncomeLoss data points:", len(net_income_points))
print("fields on one data point:", list(net_income_points[0].keys()))

# Group data points by the reporting period they describe (start, end), as opposed
# to when they were filed. More than one entry for the same period, with different
# "filed" dates, is direct evidence that a restatement is kept as an additional
# historical fact rather than overwriting the original.
from collections import defaultdict

by_period = defaultdict(list)
for point in net_income_points:
    key = (point.get("start"), point["end"])
    by_period[key].append(point)

restated_periods = {k: v for k, v in by_period.items() if len(v) > 1}
print("periods with more than one reported value:", len(restated_periods))

# Inspect one such period in full, if any exist, to see what actually differs
# between the two entries (value, filed date, which filing/form each came from).
if restated_periods:
    example_key, example_points = next(iter(restated_periods.items()))
    print("example period:", example_key)
    for p in example_points:
        print(" filed:", p["filed"], "val:", p["val"], "form:", p["form"], "accn:", p["accn"])


total NetIncomeLoss data points: 338
fields on one data point: ['start', 'end', 'val', 'accn', 'fy', 'fp', 'form', 'filed']
periods with more than one reported value: 112
example period: ('2006-10-01', '2007-09-29')
 filed: 2009-10-27 val: 3496000000 form: 10-K accn: 0001193125-09-214859
 filed: 2010-01-25 val: 3495000000 form: 10-K/A accn: 0001193125-10-012091


In [6]:
# 112 periods reported more than once does not mean 112 restatements. A company
# re-reports the same historical figure as a "prior year" comparative in every later
# filing that covers that period, identical value, only the filed date and accession
# differ. Only entries where the value itself changed are genuine restatements; the
# rest are the same number being carried forward through later filings.
genuine_restatements = {
    period: points
    for period, points in restated_periods.items()
    if len({p["val"] for p in points}) > 1
}
print("periods with more than one reported value:", len(restated_periods))
print("periods where the value actually changed:", len(genuine_restatements))

# A handful of examples: how large the correction is and how far apart the filed
# dates are, both relevant to whether this needs careful modeling or is small enough
# to treat as noise.
for period, points in list(genuine_restatements.items())[:5]:
    ordered = sorted(points, key=lambda p: p["filed"])
    print(period, [(p["filed"], p["val"], p["form"]) for p in ordered])


periods with more than one reported value: 112
periods where the value actually changed: 5
('2006-10-01', '2007-09-29') [('2009-10-27', 3496000000, '10-K'), ('2010-01-25', 3495000000, '10-K/A')]
('2007-09-30', '2008-09-27') [('2009-10-27', 4834000000, '10-K'), ('2010-01-25', 6119000000, '10-K/A'), ('2010-10-27', 6119000000, '10-K')]
('2008-09-28', '2009-06-27') [('2009-07-22', 4039000000, '10-Q'), ('2010-07-21', 5703000000, '10-Q')]
('2009-03-29', '2009-06-27') [('2009-07-22', 1229000000, '10-Q'), ('2010-07-21', 1828000000, '10-Q'), ('2010-10-27', 1828000000, '10-K')]
('2008-09-28', '2009-09-26') [('2009-10-27', 5704000000, '10-K'), ('2010-01-25', 8235000000, '10-K/A'), ('2010-10-27', 8235000000, '10-K'), ('2011-10-26', 8235000000, '10-K')]


In [7]:
def fact_as_of(facts, taxonomy, tag, unit, period_end, as_of_date):
    # The value known as of as_of_date for the period ending on period_end: the
    # entry with the latest "filed" date that is still <= as_of_date. Mirrors
    # ticker_on()'s "most recent thing true as of this date" pattern in
    # src/universe/point_in_time.py, applied to a filed fact instead of a
    # membership span.
    points = facts["facts"][taxonomy][tag]["units"][unit]
    candidates = [p for p in points if p["end"] == period_end and p["filed"] <= as_of_date]
    if not candidates:
        return None  # nothing had been filed yet as of this date
    latest = max(candidates, key=lambda p: p["filed"])
    return latest["val"], latest["filed"], latest["form"]

# Test directly against the real FY2008 restatement found above. A naive fundamentals
# loader that just reads "the current number" would return the post-restatement
# value even for a 2009 backtest date, a straightforward look-ahead violation.
period_end = "2008-09-27"
print("as of 2009-12-01:", fact_as_of(apple_facts, "us-gaap", "NetIncomeLoss", "USD", period_end, "2009-12-01"))
print("as of 2010-06-01:", fact_as_of(apple_facts, "us-gaap", "NetIncomeLoss", "USD", period_end, "2010-06-01"))


as of 2009-12-01: (4834000000, '2009-10-27', '10-K')
as of 2010-06-01: (6119000000, '2010-01-25', '10-K/A')


In [8]:
# GrossProfit itself isn't always tagged (DoorDash has none at all). When absent, it
# can be derived as revenue minus cost of revenue, but cost of revenue's tag name is
# no more standardized than revenue's was. Extending the same alias mechanism to a
# second concept rather than inventing a new pattern for it.
TAG_ALIASES["cost_of_revenue"] = [
    "CostOfRevenue",
    "CostOfGoodsAndServicesSold",
    "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
]

def resolve_gross_profit_tags(facts):
    # Returns ("direct", tag) if GrossProfit is tagged outright, or
    # ("derived", revenue_tag, cost_tag) if it has to be computed from the two
    # components instead, or None if not even those can be found.
    gaap = facts["facts"].get("us-gaap", {})
    if "GrossProfit" in gaap:
        return ("direct", "GrossProfit")
    revenue_tag = resolve_tag(facts, "us-gaap", "revenue")
    cost_tag = resolve_tag(facts, "us-gaap", "cost_of_revenue")
    if revenue_tag and cost_tag:
        return ("derived", revenue_tag, cost_tag)
    return None

for name, cik in companies_to_check:
    facts = fetch_company_facts(cik)
    print(name, resolve_gross_profit_tags(facts))


Apple ('direct', 'GrossProfit')
Costco ('direct', 'GrossProfit')
DoorDash ('derived', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization')


In [9]:
# Revenue and cost of revenue only ever needed to search within us-gaap. Shares
# outstanding needs to search across two taxonomies (dei's cover-page tag first,
# us-gaap's balance-sheet tag as fallback), which the old resolve_tag signature
# (fixed taxonomy, tag names only) can't express. Generalizing every alias list to
# (taxonomy, tag) pairs uniformly, rather than keeping two different alias shapes
# side by side.
TAG_ALIASES = {
    "revenue": [
        ("us-gaap", "Revenues"),
        ("us-gaap", "RevenueFromContractWithCustomerExcludingAssessedTax"),
    ],
    "cost_of_revenue": [
        ("us-gaap", "CostOfRevenue"),
        ("us-gaap", "CostOfGoodsAndServicesSold"),
        ("us-gaap", "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization"),
    ],
    "shares_outstanding": [
        ("dei", "EntityCommonStockSharesOutstanding"),
        ("us-gaap", "CommonStockSharesOutstanding"),
    ],
}

def resolve_tag(facts, concept):
    # Returns the (taxonomy, tag) pair for the first alias actually present in this
    # company's facts, or None if the company reports the concept under some other
    # name not covered here (DoorDash's shares outstanding, confirmed below).
    for taxonomy, tag in TAG_ALIASES[concept]:
        if tag in facts["facts"].get(taxonomy, {}):
            return (taxonomy, tag)
    return None

# resolve_gross_profit_tags now calls the two-argument form (taxonomy is no longer
# passed separately, it's part of what resolve_tag returns).
def resolve_gross_profit_tags(facts):
    gaap = facts["facts"].get("us-gaap", {})
    if "GrossProfit" in gaap:
        return ("direct", "GrossProfit")
    revenue = resolve_tag(facts, "revenue")
    cost = resolve_tag(facts, "cost_of_revenue")
    if revenue and cost:
        return ("derived", revenue, cost)
    return None

for name, cik in companies_to_check + [("Alphabet", 1652044)]:
    facts = fetch_company_facts(cik)
    print(name, "shares_outstanding ->", resolve_tag(facts, "shares_outstanding"))


Apple shares_outstanding -> ('dei', 'EntityCommonStockSharesOutstanding')
Costco shares_outstanding -> ('dei', 'EntityCommonStockSharesOutstanding')
DoorDash shares_outstanding -> None
Alphabet shares_outstanding -> ('us-gaap', 'CommonStockSharesOutstanding')


In [10]:
# Alphabet has now been used in enough checks to belong in the standing test set
# rather than being appended ad hoc each time.
companies_to_check = [
    ("Apple", 320193),
    ("Costco", 909832),
    ("DoorDash", 1792789),
    ("Alphabet", 1652044),
]

# DoorDash's long-term debt was not a naming variant of the same concept, it is
# structurally different debt (convertible notes rather than traditional term debt),
# tagged accordingly. Total liabilities of $9.5B (checked separately) confirmed this
# is a real gap to fill, not a genuinely zero balance.
TAG_ALIASES["long_term_debt_noncurrent"] = [
    ("us-gaap", "LongTermDebtNoncurrent"),
    ("us-gaap", "ConvertibleLongTermNotesPayable"),
]
TAG_ALIASES["long_term_debt_current"] = [
    ("us-gaap", "LongTermDebtCurrent"),
    ("us-gaap", "ConvertibleNotesPayableCurrent"),
]

for name, cik in companies_to_check:
    facts = fetch_company_facts(cik)
    noncurrent = resolve_tag(facts, "long_term_debt_noncurrent")
    current = resolve_tag(facts, "long_term_debt_current")
    print(name, "noncurrent ->", noncurrent, "| current ->", current)


Apple noncurrent -> ('us-gaap', 'LongTermDebtNoncurrent') | current -> ('us-gaap', 'LongTermDebtCurrent')
Costco noncurrent -> ('us-gaap', 'LongTermDebtNoncurrent') | current -> ('us-gaap', 'LongTermDebtCurrent')
DoorDash noncurrent -> ('us-gaap', 'ConvertibleLongTermNotesPayable') | current -> ('us-gaap', 'ConvertibleNotesPayableCurrent')
Alphabet noncurrent -> ('us-gaap', 'LongTermDebtNoncurrent') | current -> ('us-gaap', 'LongTermDebtCurrent')


In [11]:
# Wires resolve_tag and fact_as_of together: given a concept name, this figures out
# which tag this specific filer uses for it, then returns the value known as of a
# given date, in one call. Previously these were tested separately; every theme
# built from here on needs both steps together, so it belongs in one function.
TAG_ALIASES["net_income"] = [("us-gaap", "NetIncomeLoss")]
TAG_ALIASES["operating_cash_flow"] = [("us-gaap", "NetCashProvidedByUsedInOperatingActivities")]
TAG_ALIASES["total_assets"] = [("us-gaap", "Assets")]

def concept_value_as_of(facts, concept, unit, period_end, as_of_date):
    resolved = resolve_tag(facts, concept)
    if resolved is None:
        return None  # this filer doesn't report the concept under any alias tried yet
    taxonomy, tag = resolved
    return fact_as_of(facts, taxonomy, tag, unit, period_end, as_of_date)

# Accruals (cash-flow method): the non-cash share of earnings, scaled by assets.
# Low or negative accruals (cash flow exceeding reported earnings) is the "higher
# quality earnings" end of the anomaly; the balance-sheet method (change in
# receivables/inventory/payables) is the more standard version in the literature but
# pulls in less standardized tags, so this cash-flow version is the first cut.
# One real annual period per company, chosen to fall shortly after each one's actual
# 10-K filing date.
cases = [
    ("Apple", 320193, "2020-09-26", "2020-12-01"),
    ("Costco", 909832, "2020-08-30", "2020-11-01"),
    ("DoorDash", 1792789, "2021-12-31", "2022-03-01"),
    ("Alphabet", 1652044, "2020-12-31", "2021-03-01"),
]
for name, cik, period_end, as_of in cases:
    facts = fetch_company_facts(cik)
    ni = concept_value_as_of(facts, "net_income", "USD", period_end, as_of)
    cfo = concept_value_as_of(facts, "operating_cash_flow", "USD", period_end, as_of)
    assets = concept_value_as_of(facts, "total_assets", "USD", period_end, as_of)
    if ni and cfo and assets:
        accruals = (ni[0] - cfo[0]) / assets[0]
        print(name, "accruals:", accruals)
    else:
        print(name, "missing a required value:", ni, cfo, assets)


Apple accruals: -0.07182421083831447
Costco accruals: -0.08746130030959752
DoorDash accruals: -0.17036275517697166
Alphabet accruals: -0.07776519323187825


In [12]:
# concept_value_as_of previously called resolve_tag once to pick a single tag for a
# company, then only ever queried that one tag. Alphabet's 2021-2022 debt data
# (found under a completely different, undifferentiated "LongTermDebt" tag that
# year) showed this assumption is wrong: which tag a company uses can change across
# its own filing history, not just vary company to company. Rewritten to try every
# alias for the specific period being asked about, taking the first one with an
# actual value, rather than committing to one tag for the whole company up front.
def concept_value_as_of(facts, concept, unit, period_end, as_of_date):
    for taxonomy, tag in TAG_ALIASES[concept]:
        points = facts["facts"].get(taxonomy, {}).get(tag, {}).get("units", {}).get(unit, [])
        candidates = [p for p in points if p["end"] == period_end and p["filed"] <= as_of_date]
        if candidates:
            latest = max(candidates, key=lambda p: p["filed"])
            return latest["val"], latest["filed"], latest["form"], tag
    # No alias had a value for this exact period, as of this date. Returned as None,
    # never inferred as zero: a missing tag might mean zero debt, or might mean the
    # breakdown simply wasn't filed yet (confirmed directly with Alphabet's 2019
    # current-debt figure, filed 10 months after the original 10-K). Conflating the
    # two would misrepresent what was actually known on a given date.
    return None

# A third alias for long term debt, found only by tracing why Alphabet's 2021 and
# 2022 figures came back missing: some filers report one undifferentiated
# "LongTermDebt" figure in certain years instead of splitting noncurrent and current.
TAG_ALIASES["long_term_debt_noncurrent"].append(("us-gaap", "LongTermDebt"))

def total_debt_as_of(facts, period_end, as_of_date):
    noncurrent = concept_value_as_of(facts, "long_term_debt_noncurrent", "USD", period_end, as_of_date)
    current = concept_value_as_of(facts, "long_term_debt_current", "USD", period_end, as_of_date)
    if noncurrent is None or current is None:
        return None  # propagate missing rather than guess a component is zero
    return noncurrent[0] + current[0]

# Debt issuance: year over year change in total long term debt, scaled by prior
# assets. Same convenience sample as before.
cases = [
    ("Apple", 320193, "2019-09-28", "2019-12-01", "2020-09-26", "2020-12-01"),
    ("DoorDash", 1792789, "2020-12-31", "2021-03-01", "2021-12-31", "2022-03-01"),
    ("Alphabet", 1652044, "2019-12-31", "2020-03-01", "2020-12-31", "2021-03-01"),
]
for name, cik, prior_end, prior_as_of, curr_end, curr_as_of in cases:
    facts = fetch_company_facts(cik)
    prior_debt = total_debt_as_of(facts, prior_end, prior_as_of)
    curr_debt = total_debt_as_of(facts, curr_end, curr_as_of)
    prior_assets = concept_value_as_of(facts, "total_assets", "USD", prior_end, prior_as_of)
    if prior_debt is not None and curr_debt is not None and prior_assets:
        print(name, "debt issuance:", (curr_debt - prior_debt) / prior_assets[0])
    else:
        print(name, "debt issuance: unresolved | prior debt:", prior_debt, "| current debt:", curr_debt)


Apple debt issuance: 0.015872218742984084
DoorDash debt issuance: unresolved | prior debt: None | current debt: None
Alphabet debt issuance: unresolved | prior debt: None | current debt: 16318000000


In [13]:
# Three more themes, all reusing tags and mechanics already validated by accruals
# and debt issuance: no new resolution logic needed, just new TAG_ALIASES entries
# and new formulas.
TAG_ALIASES["total_liabilities"] = [("us-gaap", "Liabilities")]
TAG_ALIASES["stockholders_equity"] = [("us-gaap", "StockholdersEquity")]

# DoorDash's dates below are shifted a few days later than the pattern used for
# other companies (2021-04-01 instead of 2021-03-01): its actual FY2020 10-K wasn't
# filed until 2021-03-05, so the earlier date correctly returned "unresolved" rather
# than skip-ahead data, a real example of the point-in-time rule doing its job
# rather than a bug.
cases = [
    ("Apple", 320193, "2019-09-28", "2019-12-01", "2020-09-26", "2020-12-01"),
    ("Costco", 909832, "2019-09-01", "2019-11-01", "2020-08-30", "2020-11-01"),
    ("DoorDash", 1792789, "2020-12-31", "2021-04-01", "2021-12-31", "2022-03-15"),
    ("Alphabet", 1652044, "2019-12-31", "2020-03-01", "2020-12-31", "2021-03-01"),
]

for name, cik, prior_end, prior_as_of, curr_end, curr_as_of in cases:
    facts = fetch_company_facts(cik)
    prior_assets = concept_value_as_of(facts, "total_assets", "USD", prior_end, prior_as_of)
    curr_assets = concept_value_as_of(facts, "total_assets", "USD", curr_end, curr_as_of)
    prior_ni = concept_value_as_of(facts, "net_income", "USD", prior_end, prior_as_of)
    curr_ni = concept_value_as_of(facts, "net_income", "USD", curr_end, curr_as_of)
    curr_liab = concept_value_as_of(facts, "total_liabilities", "USD", curr_end, curr_as_of)

    print(name)
    if prior_assets and curr_assets:
        # Investment: asset growth, the simplest of the eight themes, one tag at
        # two dates.
        print("  investment:", (curr_assets[0] - prior_assets[0]) / prior_assets[0])
    if prior_ni and curr_ni and prior_assets:
        # Profit growth: change in earnings, scaled by prior assets for the same
        # reason accruals and debt issuance were, comparable magnitude across
        # companies of very different sizes, and no sign flip when prior net
        # income itself is negative (relevant for DoorDash).
        print("  profit growth:", (curr_ni[0] - prior_ni[0]) / prior_assets[0])
    if curr_liab and curr_assets:
        # Low leverage: book leverage (liabilities / assets), a single-date ratio,
        # no year-over-year comparison needed.
        print("  low leverage:", curr_liab[0] / curr_assets[0])


Apple
  investment: -0.04321213768330005
  profit growth: 0.006366021103876921
  low leverage: 0.7982666847799239
Costco
  investment: 0.22370044052863436
  profit growth: 0.007555066079295154
  low leverage: 0.663312693498452
DoorDash
  investment: 0.071777113174878
  profit growth: -0.0011018416496143555
  low leverage: 0.31458363930092526
Alphabet
  investment: 0.1584109253413263
  profit growth: 0.02147809603891138
  low leverage: 0.3037144573488186


In [14]:
import os
import pandas as pd

# This kernel's working directory is notebooks/ (VS Code defaults to the active
# file's own directory), not the project root, which breaks "from src..." imports
# and any project-root-relative path assumption, the same assumption
# src/loaders/prices.py's own DATA_RAW constant relies on. Guarded so re-running
# this cell doesn't walk up an extra level by accident.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

from src.loaders.prices import load_cik_prices

# Value: the first theme requiring an actual join with the price loader (already
# built, src/loaders/prices.py), not just fundamentals data in isolation. Market cap
# = shares outstanding x price, then book/price and earnings/price against it.
def price_as_of(prices_df, as_of_date):
    # prices.py's cached index is tz-aware (America/New_York, per
    # loaders_construction.md's Part 1 finding); as_of_date here is a plain
    # "YYYY-MM-DD" string, same shape used everywhere in this notebook. Comparing
    # calendar dates only, not wall-clock time: the question is "which trading
    # day's close was the most recent as of this date," not an intraday one.
    as_of = pd.Timestamp(as_of_date)
    eligible = prices_df[prices_df.index.tz_localize(None).normalize() <= as_of]
    if eligible.empty:
        return None  # nothing traded yet as of this date, e.g. before this CIK's cached span started
    return eligible.iloc[-1]["Close"], eligible.index[-1]

# DoorDash is deliberately excluded from this pass: its cached prices only start
# 2025-03-31 (the price loader caches a CIK's actual S&P 500 membership span, not
# its full trading history, and DoorDash joined the index in March 2025), and its
# shares outstanding is already a known, separate gap (Part 4). Neither is a value-
# theme bug; there is simply no period where both inputs are available for it yet.
cases = [
    ("Apple", 320193, "2020-09-26", "2020-12-01"),
    ("Costco", 909832, "2020-08-30", "2020-11-01"),
    ("Alphabet", 1652044, "2020-12-31", "2021-03-01"),
]
for name, cik, period_end, as_of in cases:
    facts = fetch_company_facts(cik)
    shares = concept_value_as_of(facts, "shares_outstanding", "shares", period_end, as_of)
    equity = concept_value_as_of(facts, "stockholders_equity", "USD", period_end, as_of)
    ni = concept_value_as_of(facts, "net_income", "USD", period_end, as_of)
    prices = load_cik_prices(cik)
    price = price_as_of(prices, as_of)

    if shares and equity and ni and price:
        market_cap = shares[0] * price[0]
        print(name, "market cap:", market_cap)
        print("  book to price:", equity[0] / market_cap)
        print("  earnings to price:", ni[0] / market_cap)
    else:
        print(name, "unresolved:", shares, equity, ni, price)


/Users/hongxianli/Documents/data_science/capm-portfolio
Apple market cap: 2024317861856.0486
  book to price: 0.032277045631604634
  earnings to price: 0.02836066463759858
Costco market cap: 144857493553.16162
  book to price: 0.12622060172047578
  earnings to price: 0.027627152050171955
Alphabet market cap: 69259853979.30908
  book to price: 3.2131745479348472
  earnings to price: 0.5814190716028667


In [15]:
from src.universe.point_in_time import build_universe, ticker_on

universe_spans, ticker_history = build_universe()

def price_as_of(cik, as_of_date):
    # Some CIKs (dual class shares, e.g. Alphabet's GOOG/GOOGL) have more than one
    # concurrently traded ticker inside the same cached file. Reading the raw file
    # directly and taking "whichever row sorts last" mixes rows across tickers,
    # which picks between two genuinely different real prices arbitrarily and (see
    # split_adjustment_factor below) double counts split events. ticker_on is the
    # loader's own established way to pick one consistent ticker for a CIK on a
    # given date, so it's used here rather than reading load_cik_prices' output
    # unfiltered.
    ticker = ticker_on(ticker_history, cik, as_of_date)
    prices_df = load_cik_prices(cik)
    if ticker is None or prices_df is None:
        return None
    one_ticker = prices_df[prices_df["ticker"] == ticker]
    as_of = pd.Timestamp(as_of_date)
    eligible = one_ticker[one_ticker.index.tz_localize(None).normalize() <= as_of]
    if eligible.empty:
        return None
    return eligible.iloc[-1]["Close"], one_ticker

def split_adjustment_factor(one_ticker_prices, since_date):
    # Cached prices are always split adjusted (loaders_construction.md, Part 1: no
    # "Adj Close" column exists because the adjustment is baked into Close
    # directly), but a historical shares-outstanding figure from an SEC filing is
    # as-filed, never retroactively adjusted for a split that hadn't happened yet.
    # Combining the two directly silently understates market cap by the split
    # ratio, confirmed against Alphabet's real 20-for-1 split on 2022-07-18.
    # This is the fix: the cumulative product of every split recorded after
    # since_date, within one ticker's own series (see price_as_of above),
    # reproduces the same factor already baked into the price, so applying it to
    # the as-filed share count puts both sides on the same basis.
    since = pd.Timestamp(since_date)
    later_splits = one_ticker_prices[
        (one_ticker_prices.index.tz_localize(None).normalize() > since)
        & (one_ticker_prices["Stock Splits"] != 0)
    ]
    factor = 1.0
    for ratio in later_splits["Stock Splits"]:
        factor *= ratio
    return factor

cases = [
    ("Apple", 320193, "2020-09-26", "2020-12-01"),
    ("Costco", 909832, "2020-08-30", "2020-11-01"),
    ("Alphabet", 1652044, "2020-12-31", "2021-03-01"),
]
for name, cik, period_end, as_of in cases:
    facts = fetch_company_facts(cik)
    shares = concept_value_as_of(facts, "shares_outstanding", "shares", period_end, as_of)
    equity = concept_value_as_of(facts, "stockholders_equity", "USD", period_end, as_of)
    ni = concept_value_as_of(facts, "net_income", "USD", period_end, as_of)
    price_result = price_as_of(cik, as_of)

    if shares and equity and ni and price_result:
        price, one_ticker = price_result
        factor = split_adjustment_factor(one_ticker, period_end)
        market_cap = shares[0] * factor * price
        print(name, "split factor:", factor, "| market cap:", market_cap)
        print("  book to price:", equity[0] / market_cap)
        print("  earnings to price:", ni[0] / market_cap)
    else:
        print(name, "unresolved:", shares, equity, ni, price_result)


Apple split factor: 1.0 | market cap: 2024317861856.0486
  book to price: 0.032277045631604634
  earnings to price: 0.02836066463759858
Costco split factor: 1.0 | market cap: 144857493553.16162
  book to price: 0.12622060172047578
  earnings to price: 0.027627152050171955
Alphabet split factor: 20.0 | market cap: 1385197079586.1816
  book to price: 0.16065872739674236
  earnings to price: 0.029070953580143336


In [16]:
import random
import time
from src.universe.point_in_time import build_universe

# Everything up to now has been tested against 4 hand-picked companies chosen to
# surface known-hard cases. That says nothing about the general failure rate across
# the actual point in time universe. A real, if modest, sample: 30 companies drawn
# from the actual S&P 500 membership on a real date, not hand-picked.
universe_spans, ticker_history = build_universe()
active = universe_spans[
    (universe_spans["start_date"] <= "2020-01-01")
    & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= "2020-01-01"))
]
sample = active.sample(30, random_state=42)

def resolve_anywhere(facts, concept):
    # Lighter than concept_value_as_of: just "does this filer report the concept
    # under any known alias, anywhere in its history," not a specific dated value.
    # Enough to measure coverage without a full point in time query per company.
    for taxonomy, tag in TAG_ALIASES[concept]:
        if tag in facts["facts"].get(taxonomy, {}):
            return True
    return False

results = {concept: 0 for concept in TAG_ALIASES}
n = 0
for _, row in sample.iterrows():
    facts = fetch_company_facts(int(row["cik"]))
    n += 1
    for concept in TAG_ALIASES:
        if resolve_anywhere(facts, concept):
            results[concept] += 1
    time.sleep(0.15)  # a courtesy pause; SEC's own rate-limit guidance is a fair-use ask, not enforced per-request

for concept, count in results.items():
    print(f"{concept}: {count}/{n} ({100*count/n:.0f}%)")


revenue: 27/30 (90%)
cost_of_revenue: 20/30 (67%)
shares_outstanding: 30/30 (100%)
long_term_debt_noncurrent: 27/30 (90%)
long_term_debt_current: 18/30 (60%)
net_income: 29/30 (97%)
operating_cash_flow: 29/30 (97%)
total_assets: 30/30 (100%)
total_liabilities: 22/30 (73%)
stockholders_equity: 29/30 (97%)


In [17]:
TAG_ALIASES["total_liabilities"] = [("us-gaap", "Liabilities")]

def total_liabilities_as_of(facts, period_end, as_of_date):
    # Liabilities itself isn't always tagged (confirmed: 8 of 30 sampled S&P 500
    # companies omit it despite reporting Assets and StockholdersEquity directly),
    # not because the data is missing, but because it's implied by the balance
    # sheet identity Liabilities = Assets - StockholdersEquity and some filers
    # simply don't tag the derived total explicitly. Same shape as GrossProfit's
    # direct-vs-derived split in Part 2.
    direct = concept_value_as_of(facts, "total_liabilities", "USD", period_end, as_of_date)
    if direct is not None:
        return direct[0]
    assets = concept_value_as_of(facts, "total_assets", "USD", period_end, as_of_date)
    equity = concept_value_as_of(facts, "stockholders_equity", "USD", period_end, as_of_date)
    if assets is not None and equity is not None:
        return assets[0] - equity[0]
    return None


In [18]:
import pandas as pd
# Sanity checks against synthetic fixtures, not live data. Two of the mechanisms
# built in this notebook already hid real bugs behind "looks right for every
# company tried so far" (concept_value_as_of's original per-company resolution
# missed Alphabet's mid-history tag switch; the first split_adjustment_factor
# attempt silently double counted a dual-ticker split). Fixtures make each exact
# edge case explicit, rather than relying on finding the right real company again.
# Every synthetic fact below includes "form", since concept_value_as_of unpacks it
# unconditionally, matching the shape every real EDGAR fact always has.

# concept_value_as_of must resolve per period, not lock onto one tag per company.
# Mirrors Alphabet's real history: LongTermDebtNoncurrent used through 2020,
# LongTermDebt used instead for 2021, both are real aliases already in TAG_ALIASES.
switch_facts = {"facts": {"us-gaap": {
    "LongTermDebtNoncurrent": {"units": {"USD": [{"end": "2020-12-31", "val": 500, "filed": "2021-02-01", "form": "10-K"}]}},
    "LongTermDebt": {"units": {"USD": [{"end": "2021-12-31", "val": 700, "filed": "2022-02-01", "form": "10-K"}]}},
}}}
assert concept_value_as_of(switch_facts, "long_term_debt_noncurrent", "USD", "2020-12-31", "2021-06-01")[0] == 500
assert concept_value_as_of(switch_facts, "long_term_debt_noncurrent", "USD", "2021-12-31", "2022-06-01")[0] == 700

# concept_value_as_of must return the latest filed date on or before as_of_date,
# never a later one, and must return None before anything was filed at all.
# Mirrors Apple's real FY2008 restatement (Part 3).
restated_facts = {"facts": {"us-gaap": {"NetIncomeLoss": {"units": {"USD": [
    {"end": "2020-12-31", "val": 100, "filed": "2021-02-01", "form": "10-K"},
    {"end": "2020-12-31", "val": 150, "filed": "2021-06-01", "form": "10-K/A"},
]}}}}}
assert concept_value_as_of(restated_facts, "net_income", "USD", "2020-12-31", "2021-03-01")[0] == 100
assert concept_value_as_of(restated_facts, "net_income", "USD", "2020-12-31", "2021-07-01")[0] == 150
assert concept_value_as_of(restated_facts, "net_income", "USD", "2020-12-31", "2021-01-01") is None

# total_liabilities_as_of: direct tag wins when present; falls back to
# Assets - StockholdersEquity when absent; None when neither can be resolved.
direct_facts = {"facts": {"us-gaap": {"Liabilities": {"units": {"USD": [{"end": "2020-12-31", "val": 900, "filed": "2021-02-01", "form": "10-K"}]}}}}}
assert total_liabilities_as_of(direct_facts, "2020-12-31", "2021-03-01") == 900

derived_facts = {"facts": {"us-gaap": {
    "Assets": {"units": {"USD": [{"end": "2020-12-31", "val": 1000, "filed": "2021-02-01", "form": "10-K"}]}},
    "StockholdersEquity": {"units": {"USD": [{"end": "2020-12-31", "val": 300, "filed": "2021-02-01", "form": "10-K"}]}},
}}}
assert total_liabilities_as_of(derived_facts, "2020-12-31", "2021-03-01") == 700

empty_facts = {"facts": {"us-gaap": {}}}
assert total_liabilities_as_of(empty_facts, "2020-12-31", "2021-03-01") is None

# split_adjustment_factor: multiplies every split strictly after since_date, and
# excludes one that lands on or before it. Two splits after, at different dates,
# to check the cumulative product isn't just picking one.
fake_prices = pd.DataFrame(
    {"Close": [1, 1, 1], "Stock Splits": [4.0, 0.0, 2.0]},
    index=pd.to_datetime(["2020-01-01", "2020-06-01", "2021-01-01"]).tz_localize("America/New_York"),
)
assert split_adjustment_factor(fake_prices, "2019-12-31") == 8.0  # both splits count
assert split_adjustment_factor(fake_prices, "2020-01-01") == 2.0  # the first split excluded (on, not after)
assert split_adjustment_factor(fake_prices, "2021-01-01") == 1.0  # nothing left after this date

print("all sanity checks passed")


all sanity checks passed


In [19]:
import time

from src.loaders.fundamentals import TAG_ALIASES, fetch_company_facts
from src.universe.point_in_time import build_universe

# A broader, more representative sample than the earlier 4 and 30 company
# tests: mixes companies still in the index today with ones that have left it
# (acquired, delisted, bankrupt), since removed companies are exactly what
# point in time backtesting needs most and what a single current snapshot
# never covers. Also naturally spans different eras, which is what surfaced
# MeadWestvaco's pre-2018 SalesRevenueNet tag in the last round: a
# same-day-only sample can only ever see whichever accounting-standard era
# happens to be current.
universe_spans, ticker_history = build_universe()

left_index = universe_spans[universe_spans["end_date"].notna() & universe_spans["cik"].notna()]
removed_sample = left_index.drop_duplicates("cik").sample(30, random_state=11)

current = universe_spans[
    (universe_spans["start_date"] <= "2025-01-01")
    & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= "2025-01-01"))
]
current_sample = current.dropna(subset=["cik"]).sample(30, random_state=11)

combined = list(zip(removed_sample["ticker"], removed_sample["cik"])) + list(
    zip(current_sample["ticker"], current_sample["cik"])
)
combined = list({int(cik): ticker for ticker, cik in combined}.items())
print("sample size:", len(combined))

# Fetched once and kept in memory for the rest of the session, so later cells
# can run different checks against the same sample without re-fetching.
by_ticker = {}
for cik, ticker in combined:
    try:
        by_ticker[ticker] = fetch_company_facts(cik)
    except Exception:
        by_ticker[ticker] = None
    time.sleep(0.15)

fetched = {t: f for t, f in by_ticker.items() if f is not None}
print("fetched successfully:", len(fetched), "/", len(by_ticker))


sample size: 60
fetched successfully: 60 / 60


In [20]:
def resolve_anywhere(facts, concept):
    # Existence only, not a dated value: enough to measure whether a concept
    # is reachable under any known alias at all, before worrying about
    # whether a specific period resolves.
    for taxonomy, tag in TAG_ALIASES[concept]:
        if tag in facts["facts"].get(taxonomy, {}):
            return True
    return False

# Revenue and gross profit are reported separately on purpose: "no revenue"
# and "has revenue but no cost-of-revenue path" are different findings (one
# is a missing alias, the other is a sector where the concept doesn't apply
# at all), conflating them was a real mistake made earlier in this notebook.
no_revenue = [t for t, f in fetched.items() if not resolve_anywhere(f, "revenue")]
no_gross_profit_path = [
    t for t, f in fetched.items()
    if resolve_anywhere(f, "revenue")
    and not resolve_anywhere(f, "gross_profit")
    and not resolve_anywhere(f, "cost_of_revenue")
]
no_liabilities_path = [
    t for t, f in fetched.items()
    if not resolve_anywhere(f, "total_liabilities")
    and not (resolve_anywhere(f, "total_assets") and resolve_anywhere(f, "stockholders_equity"))
]
no_shares_at_all = [
    t for t, f in fetched.items()
    if "EntityCommonStockSharesOutstanding" not in f["facts"].get("dei", {})
    and "CommonStockSharesOutstanding" not in f["facts"].get("us-gaap", {})
]

print("no revenue at all:", no_revenue)
print("has revenue, no gross profit path:", no_gross_profit_path)
print("no liabilities path at all:", no_liabilities_path)
print("no shares outstanding tag at all:", no_shares_at_all)


no revenue at all: ['PLL', 'CCU-200807', 'CCE', 'BCR', 'LLTC', 'ACT']
has revenue, no gross profit path: ['EHC', 'H-200107', 'CEG', 'KSU', 'TE', 'GGP', 'BHF', 'SEG-200011', 'INVH', 'CMG', 'UDR', 'TPL', 'BEN', 'KKR', 'ACGL']
no liabilities path at all: ['CCU-200807']
no shares outstanding tag at all: ['H-200107', 'META']


In [ ]:
# Drill-in helper: once a company shows up in one of the failure lists above,
# use this to see what it actually reports instead, the same way MeadWestvaco's
# SalesRevenueNet and Goldman's RevenuesNetOfInterestExpense were found. Pass
# a keyword (e.g. "Revenue", "Cost", "SharesOutstanding", "Debt") and it prints
# every matching tag name for that company across every taxonomy.
def dump_tags_like(ticker, keyword):
    facts = fetched[ticker]
    print(ticker, "-", facts.get("entityName"))
    for taxonomy, tags in facts["facts"].items():
        matches = sorted(t for t in tags if keyword.lower() in t.lower())
        if matches:
            print(f"  [{taxonomy}]", matches)

# Example: dump_tags_like("CMG", "Cost")
